In [7]:
import pandas as pd

df = pd.read_parquet('final_feature_matrix.parquet')
df

,TradingDate,Symbol,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,factor_bias_10d,...,factor_turnover_chg,factor_illiquidity,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret
0,2010-02-03,000001,-0.015207,-0.030839,-0.038687,-0.082833,0.030839,0.151023,0.249998,-0.025758,...,0.830036,0.000986,0.801502,-0.979179,0.416757,0.157150,-3.173486e+06,0.618633,0.0,-0.016972
1,2010-02-04,000001,0.063089,0.030580,0.056128,-0.013974,-0.030580,0.644905,0.642860,0.023805,...,1.369481,0.004663,0.955794,-0.979859,0.469072,1.000000,6.426087e+05,1.167729,0.0,-0.009050
2,2010-02-05,000001,0.038840,0.020690,-0.021164,-0.019867,-0.020690,0.489803,0.519484,0.008770,...,1.157583,0.002492,0.934634,-0.979382,0.471680,0.431843,5.200428e+05,0.648868,0.0,-0.020203
3,2010-02-08,000001,-0.025687,0.013826,-0.046794,-0.026548,-0.013826,0.476197,0.454547,0.004612,...,1.147097,0.001498,0.893853,-0.979644,0.472304,0.648357,3.349373e+05,0.591208,0.0,0.020203
4,2010-02-09,000001,-0.028829,0.015066,-0.028391,-0.046017,-0.015066,0.350010,0.336844,-0.012639,...,0.729167,0.004473,0.783953,-0.979342,0.484571,0.174614,-7.790730e+04,0.453724,0.0,0.012646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2334364,2024-12-24,688981,0.113998,0.167853,0.096796,0.104689,-0.167853,0.776543,0.776543,0.097847,...,3.060301,0.000068,1.909707,0.013404,0.651586,0.282143,2.113348e+07,2.156261,0.0,0.011599
2334365,2024-12-25,688981,0.027365,0.178489,0.100682,0.138190,-0.178489,0.869928,0.869928,0.105922,...,2.243137,0.000203,1.897180,-0.013648,0.690539,0.889655,2.317838e+07,1.296538,0.0,-0.012942
2334366,2024-12-26,688981,0.029523,0.174940,0.128657,0.113523,-0.174940,0.902636,0.902636,0.104735,...,2.034596,0.000121,1.963392,0.002524,0.695111,0.470760,2.168925e+07,1.379041,0.0,0.008031
2334367,2024-12-27,688981,-0.001342,0.132140,0.109671,0.110818,-0.132140,0.834857,0.834857,0.078902,...,1.055621,0.000186,1.866950,0.005330,0.690281,0.212454,2.141384e+07,0.989078,0.0,0.018090


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 定义列名
features = [col for col in df.columns if col.startswith('factor_')]

# --- 步骤 1: MAD 去极值 (截面处理版) ---
def winsorize_mad(s, n=3):
    # s 是一个 Series，代表某一天所有股票的某个因子值
    median = s.median()
    mad = (s - median).abs().median()
    # 1.4826 是为了使 MAD 在正态分布下等价于标准差
    threshold = n * 1.4826 * mad
    return s.clip(lower=median - threshold, upper=median + threshold)

print("正在执行 1/3: 截面 MAD 去极值...")

# 核心修改：使用 groupby('TradingDate') 确保因子处理不引入未来函数
for col in features:
    # transform 会保持原索引顺序，将处理后的数据填回
    df[col] = df.groupby('TradingDate')[col].transform(winsorize_mad)

print("✅ 去极值完成（已按交易日截面处理）")

正在执行 1/3: 截面 MAD 去极值...
✅ 去极值完成（已按交易日截面处理）


In [16]:
import numpy as np
import pandas as pd
from scipy.linalg import lstsq

def fast_neutralize_final(df, feat_cols, industry_col='Indexcode', mv_col='CirculatedMarketValue'):
    """
    极速版行业市值中性化：全样本传入，内部自动截面处理
    """
    print(f"🚀 开始中性化处理，涉及因子数: {len(feat_cols)}, 总行数: {len(df)}")
    
    # 1. 预处理：市值取对数 + 行业 One-hot (全量数据一次性完成)
    # 使用 copy 避免 SettingWithCopyWarning
    data = df.copy()
    data['log_mv'] = np.log(data[mv_col] + 1e-9)
    
    # 生成行业哑变量 (drop_first=True 消除多重共线性)
    industry_dummies = pd.get_dummies(data[industry_col], prefix='ind', drop_first=True)
    ind_cols = industry_dummies.columns.tolist()
    
    # 将哑变量拼接到工作表中
    data = pd.concat([data, industry_dummies], axis=1)
    
    # 2. 定义每一天的回归函数 (此时内部不再需要 groupby)
    def process_sub_group(group):
        # 构造 X 矩阵: 截距(1) + 市值 + 行业哑变量
        # 必须确保 group 内数据不为空
        if len(group) < (len(ind_cols) + 2):
            return group # 样本数太少不足以回归则原样返回
            
        X = np.column_stack([
            np.ones(len(group)), 
            group['log_mv'].values, 
            group[ind_cols].values
        ])
        
        # 目标 Y 矩阵 (所有因子)
        Y = group[feat_cols].values
        
        # 使用 scipy 的 lstsq，因为它比 np.linalg.lstsq 在处理奇异矩阵时更稳健
        # 求解 Y = X * Beta + residual
        beta, _, _, _ = lstsq(X, Y)
        
        # 更新因子值为残差
        group[feat_cols] = Y - X @ beta
        return group

    # 3. 执行截面分组计算
    # 注意：这里调用 groupby，内部函数只负责矩阵运算
    result = data.groupby('TradingDate', group_keys=False).apply(process_sub_group)
    
    # 4. 返回精简后的结果，只保留原始列
    return result[df.columns]

In [7]:
df_stock = pd.read_parquet('stock_data.parquet')
df_stock

,TradingDate,Symbol,OpenPrice,ClosePrice,HighPrice,LowPrice,Volume,Amount,ChangeRatio,TotalShare,...,F081001B,F081601B,F060101C,F060201C,F060301C,F060401C,PE,PB,PCF,PS
0,2010-01-01,000001,1185.217,1185.217,1185.217,1185.217,0.0,0.000000e+00,0.00000,3105433762,...,0.114957,0.045584,-16.545267,NaN,-1.123885,-16.010482,NaN,NaN,NaN,NaN
1,2010-01-01,000415,73.966,73.966,73.966,73.966,0.0,0.000000e+00,0.00000,300335834,...,NaN,3.280567,NaN,1.211856,-0.359215,NaN,NaN,NaN,NaN,NaN
2,2010-01-01,000661,87.656,87.656,87.656,87.656,0.0,0.000000e+00,0.00000,131326570,...,-0.408958,-0.078347,3.566045,1.054451,0.411169,3.039289,NaN,NaN,NaN,NaN
3,2010-01-01,600153,82.029,82.029,82.029,82.029,0.0,0.000000e+00,0.00000,1243194856,...,0.030121,0.009714,2.619866,1.172953,0.071868,2.113886,NaN,NaN,NaN,NaN
4,2010-01-01,000008,37.304,37.304,37.304,37.304,0.0,0.000000e+00,0.00000,73653208,...,NaN,0.003900,10.222513,0.927288,0.332872,10.225908,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2601533,2024-12-31,600196,370.206,364.053,371.085,363.614,12275863.0,3.070969e+08,-0.01701,2671326465,...,0.268856,0.013926,1.427370,0.991132,0.094667,1.281921,27.818553,1.172498,NaN,1.603459
2601534,2024-12-31,601377,17.859,17.173,17.887,17.173,76172078.0,4.846399e+08,-0.03841,8635987294,...,-0.279491,0.099453,8.346555,NaN,2.300396,8.522364,27.520911,0.886677,NaN,5.537064
2601535,2024-12-31,688009,7.381,7.370,7.511,7.346,47040674.0,2.969310e+08,0.00000,10589819000,...,-0.247170,-0.152055,1.653276,1.135159,0.187732,1.419449,19.064147,1.356368,NaN,1.787460
2601536,2024-12-31,688256,684.000,658.000,688.000,658.000,7021055.0,4.694181e+09,-0.03731,417456753,...,NaN,2.082956,NaN,0.678117,-2.511924,NaN,NaN,47.942255,184.799794,387.216998


In [ ]:
# 1. 确保两个 DataFrame 的日期格式一致，避免合并失败
df['TradingDate'] = pd.to_datetime(df['TradingDate'])
df_stock['TradingDate'] = pd.to_datetime(df_stock['TradingDate'])

# 2. 确保 df_stock 中没有重复的 (日期, 代码) 组合
df_stock = df_stock.drop_duplicates(subset=['TradingDate', 'Symbol'])

# 3. 使用左连接（left merge）将流通市值列添加到 df 中
# 我们只取出 df_stock 中需要的列，节省内存
df = pd.merge(
    df, 
    df_stock[['TradingDate', 'Symbol', 'CirculatedMarketValue']], 
    on=['TradingDate', 'Symbol'], 
    how='left'
)

# 4. 检查合并后的结果
# 看看是否有 Symbol 因为两边命名不一致（如 000001 vs 000001.SZ）而产生的缺失值
nan_count = df['CirculatedMarketValue'].isna().sum()
if nan_count > 0:
    print(f"⚠️ 警告：合并后发现 {nan_count} 行数据缺失流通市值，请检查代码格式是否完全一致。")
else:
    print("✅ 流通市值列已成功添加到 df。")

# 5. 为了保持习惯，重新按日期和代码排序
df = df.sort_values(['TradingDate', 'Symbol']).reset_index(drop=True)

✅ 流通市值列已成功添加到 df。
  TradingDate  Symbol  factor_mom_2d  factor_mom_5d  factor_mom_10d  \
0  2010-02-03  000001      -0.015207      -0.030839       -0.038687   
1  2010-02-03  000002      -0.006424       0.006507       -0.077535   
2  2010-02-03  000009      -0.014375       0.016819       -0.131009   
3  2010-02-03  000012      -0.060275      -0.029430       -0.151412   
4  2010-02-03  000021      -0.064412      -0.008124       -0.141445   

   factor_mom_20d  factor_reversal_5d  factor_pos_10d  factor_pos_20d  \
0       -0.082833            0.030839        0.151023        0.249998   
1       -0.104248           -0.006507        0.188112        0.125824   
2       -0.026513           -0.016819        0.136026        0.131416   
3       -0.101155            0.029430        0.035433        0.035433   
4       -0.089549            0.008124        0.029329        0.022467   

   factor_bias_10d  ...  factor_illiquidity  factor_br  factor_avg_px_bias  \
0        -0.025758  ...            0.0

In [11]:
# 1. 确保日期格式一致
df['TradingDate'] = pd.to_datetime(df['TradingDate'])
df_stock['TradingDate'] = pd.to_datetime(df_stock['TradingDate'])

# 2. 合并 Indexcode 列
# 如果 df_stock 里已经包含了 Indexcode，我们直接合并
print("正在添加 Indexcode 列...")
df = pd.merge(
    df, 
    df_stock[['TradingDate', 'Symbol', 'Indexcode']], 
    on=['TradingDate', 'Symbol'], 
    how='left'
)

# 3. 检查是否有缺失
nan_index = df['Indexcode'].isna().sum()
if nan_index > 0:
    print(f"⚠️ 提示：有 {nan_index} 行数据未匹配到 Indexcode。")
else:
    print("✅ Indexcode 已成功添加。")

# 4. 再次整理排序，保持数据整洁
df = df.sort_values(['TradingDate', 'Symbol']).reset_index(drop=True)

# 查看结果前几行
df.head(10)

正在添加 Indexcode 列...
✅ Indexcode 已成功添加。


,TradingDate,Symbol,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,factor_bias_10d,...,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret,CirculatedMarketValue,Indexcode
0,2010-02-03,000001,-0.015207,-0.030839,-0.038687,-0.082833,0.030839,0.151023,0.249998,-0.025758,...,0.801502,-0.979179,0.416757,0.157150,-3.173486e+06,0.618633,0.0,-0.016972,6.602650e+10,06010101
1,2010-02-03,000002,-0.006424,0.006507,-0.077535,-0.104248,-0.006507,0.188112,0.125824,-0.018820,...,0.682346,-0.988340,0.326537,0.000000,-4.566430e+06,0.647445,0.0,-0.004242,1.002661e+11,06040101
2,2010-02-03,000009,-0.014375,0.016819,-0.131009,-0.026513,-0.016819,0.136026,0.131416,-0.043535,...,1.023042,-0.764662,0.600236,0.109643,2.645609e+06,0.471858,0.0,0.095745,1.052351e+10,06040101
3,2010-02-03,000012,-0.060275,-0.029430,-0.151412,-0.101155,0.029430,0.035433,0.035433,-0.060120,...,0.882091,-0.903140,0.474738,0.179437,-1.059468e+06,0.798834,0.0,-0.003327,1.350512e+10,01010201
4,2010-02-03,000021,-0.064412,-0.008124,-0.141445,-0.089549,0.008124,0.029329,0.022467,-0.049619,...,0.968696,-0.886104,0.526818,0.081712,-3.485553e+05,0.421700,0.0,0.009686,1.083309e+10,07020102
5,2010-02-03,000024,0.011907,0.068000,-0.059868,-0.067515,-0.068000,0.538769,0.404702,0.017705,...,0.838720,-0.776674,0.561934,0.025641,-9.352952e+05,1.097471,0.0,-0.018506,1.760300e+10,06040101
6,2010-02-03,000027,0.000000,0.019653,-0.091239,-0.070896,-0.019653,0.188661,0.181784,-0.010415,...,0.863901,-0.844989,0.408135,0.310345,-8.185899e+05,0.770260,0.0,-0.007898,7.608933e+09,09010101
7,2010-02-03,000031,-0.011593,0.007520,-0.130678,-0.126631,-0.007520,0.100565,0.074218,-0.038147,...,0.769026,-0.923616,0.593620,0.025646,-2.226482e+06,0.777053,0.0,0.013501,1.733908e+10,06040101
8,2010-02-03,000039,0.026297,0.048507,-0.055144,0.058781,-0.048507,0.538769,0.445891,0.019890,...,1.199990,-0.952238,0.623841,0.446296,4.106158e+06,0.899953,0.0,-0.007729,3.073562e+10,02010602
9,2010-02-03,000046,-0.001631,0.017413,-0.068341,-0.081588,-0.017413,0.274067,0.201078,-0.011523,...,0.884248,-0.926586,0.539642,0.020378,-5.999061e+05,0.593206,0.0,-0.000797,5.569810e+09,06040101


In [17]:
print("正在执行 2/3: 截面行业与市值中性化 (可能耗时较长)...")
# --- 正确的执行方式 ---
# 直接传入整个 df，不要在外面写 groupby
df = fast_neutralize_final(df, features)

正在执行 2/3: 截面行业与市值中性化 (可能耗时较长)...
🚀 开始中性化处理，涉及因子数: 84, 总行数: 2334369


C:\Users\HP\AppData\Local\Temp\ipykernel_26872\3506507702.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = data.groupby('TradingDate', group_keys=False).apply(process_sub_group)


In [18]:
# --- 步骤 3: 截面标准化 (Z-Score) ---
print("正在执行 3/3: 截面标准化...")
df[features] = df.groupby('TradingDate')[features].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))

# --- 保存结果 ---
df.to_parquet('final_feature_matrix_polished.parquet', index=False)
print("--- 预处理全部完成 ---")

正在执行 3/3: 截面标准化...
--- 预处理全部完成 ---


In [19]:
df = pd.read_parquet('final_feature_matrix_polished.parquet')
df

,TradingDate,Symbol,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,factor_bias_10d,...,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret,CirculatedMarketValue,Indexcode
0,2010-02-03,000001,-0.419412,0.613713,0.627402,0.541997,-0.613713,-0.236243,0.939585,0.263445,...,0.319372,-2.734753,-0.452381,-0.795573,0.477897,-0.238663,0.0,-0.016972,6.602650e+10,06010101
1,2010-02-03,000002,0.120885,0.268189,0.435331,-0.229261,-0.268189,-0.007128,-0.321460,0.465881,...,-0.548896,-0.467890,-1.799874,-0.564989,-1.929665,-0.263356,0.0,-0.004242,1.002661e+11,06040101
2,2010-02-03,000009,-0.476138,0.544517,-0.632029,0.934315,-0.544517,-0.551340,-0.479064,-0.412201,...,1.264516,0.034123,1.131501,-0.054876,2.464601,-0.984421,0.0,0.095745,1.052351e+10,06040101
3,2010-02-03,000012,-1.256229,-1.338249,-1.340817,-0.825749,1.338249,-1.678090,-1.466028,-1.710890,...,0.123518,-1.273525,-0.079102,0.248499,-0.857317,-0.397479,0.0,-0.003327,1.350512e+10,01010201
4,2010-02-03,000021,-0.355677,0.674963,0.227034,0.355731,-0.674963,-0.187312,-0.187903,0.269665,...,0.585947,-0.304910,0.440726,-0.199462,-0.528501,0.073503,0.0,0.009686,1.083309e+10,07020102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2334364,2024-12-30,688472,-0.053164,-0.020868,-0.579479,-0.265367,0.020868,-0.719673,-0.315587,-0.123546,...,0.897657,2.342325,-1.255127,0.252305,-1.113922,-2.094955,0.0,0.030727,1.683167e+10,20301010
2334365,2024-12-30,688506,2.798698,0.225661,0.466909,-0.980781,-0.225661,0.860256,-0.013906,0.174716,...,0.825895,2.890364,-0.182732,1.066892,0.074647,0.811098,0.0,-0.005098,1.743081e+10,35202020
2334366,2024-12-30,688561,-0.840162,-0.387732,0.081418,-0.498613,0.387732,-0.277653,-0.325313,-0.554078,...,-0.297715,1.837282,-0.080424,-0.924835,-0.424770,0.638796,0.0,-0.051927,1.936297e+10,45102010
2334367,2024-12-30,688599,-1.202224,-2.026339,-1.993219,-0.849288,2.026339,-1.182325,-0.719093,-2.347848,...,-0.771212,2.222776,0.306306,-1.351603,0.245011,2.128963,0.0,-0.036625,4.363089e+10,20301010


In [20]:
cols_to_drop = ['Indexcode', 'CirculatedMarketValue']

# 检查这两列是否在 df 中，存在才删除，防止报错
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [21]:
df.to_parquet('final_feature_matrix_polished.parquet', index=False)

In [22]:
df

,TradingDate,Symbol,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,factor_pos_20d,factor_bias_10d,...,factor_turnover_chg,factor_illiquidity,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_log_ret
0,2010-02-03,000001,-0.419412,0.613713,0.627402,0.541997,-0.613713,-0.236243,0.939585,0.263445,...,0.167893,-0.591555,0.319372,-2.734753,-0.452381,-0.795573,0.477897,-0.238663,0.0,-0.016972
1,2010-02-03,000002,0.120885,0.268189,0.435331,-0.229261,-0.268189,-0.007128,-0.321460,0.465881,...,-0.451399,-0.548991,-0.548896,-0.467890,-1.799874,-0.564989,-1.929665,-0.263356,0.0,-0.004242
2,2010-02-03,000009,-0.476138,0.544517,-0.632029,0.934315,-0.544517,-0.551340,-0.479064,-0.412201,...,-0.182673,-0.678319,1.264516,0.034123,1.131501,-0.054876,2.464601,-0.984421,0.0,0.095745
3,2010-02-03,000012,-1.256229,-1.338249,-1.340817,-0.825749,1.338249,-1.678090,-1.466028,-1.710890,...,0.443794,-0.025056,0.123518,-1.273525,-0.079102,0.248499,-0.857317,-0.397479,0.0,-0.003327
4,2010-02-03,000021,-0.355677,0.674963,0.227034,0.355731,-0.674963,-0.187312,-0.187903,0.269665,...,-0.550184,0.844361,0.585947,-0.304910,0.440726,-0.199462,-0.528501,0.073503,0.0,0.009686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2334364,2024-12-30,688472,-0.053164,-0.020868,-0.579479,-0.265367,0.020868,-0.719673,-0.315587,-0.123546,...,-0.883087,-0.893420,0.897657,2.342325,-1.255127,0.252305,-1.113922,-2.094955,0.0,0.030727
2334365,2024-12-30,688506,2.798698,0.225661,0.466909,-0.980781,-0.225661,0.860256,-0.013906,0.174716,...,1.671173,1.730541,0.825895,2.890364,-0.182732,1.066892,0.074647,0.811098,0.0,-0.005098
2334366,2024-12-30,688561,-0.840162,-0.387732,0.081418,-0.498613,0.387732,-0.277653,-0.325313,-0.554078,...,0.126780,0.968039,-0.297715,1.837282,-0.080424,-0.924835,-0.424770,0.638796,0.0,-0.051927
2334367,2024-12-30,688599,-1.202224,-2.026339,-1.993219,-0.849288,2.026339,-1.182325,-0.719093,-2.347848,...,0.940073,1.582597,-0.771212,2.222776,0.306306,-1.351603,0.245011,2.128963,0.0,-0.036625


In [3]:
import pandas as pd
df_300 = pd.read_parquet('hs300_daily_data.parquet')
df_factor2 = pd.read_parquet('final_feature_matrix_polished.parquet')

In [4]:
# 1. 确保两个 DataFrame 的主键格式一致
df_300['TradingDate'] = pd.to_datetime(df_300['TradingDate'])
df_factor2['TradingDate'] = pd.to_datetime(df_factor2['TradingDate'])

# 2. 准备成分股映射集合 (利用 set 提高检索速度)
# 将 df_300 中的日期和代码组合成一个唯一键集合
hs300_lookup = set(zip(df_300['TradingDate'], df_300['Symbol']))

# 3. 在 df_factor2 中插入标签列
# 我们在 Symbol 之后插入 'is_hs300' 列，默认值为 False
if 'is_hs300' not in df_factor2.columns:
    symbol_pos = df_factor2.columns.get_loc('Symbol') + 1
    
    # 使用向量化映射或 apply (在大数据量下，zip+set 映射效率极高)
    df_factor2.insert(symbol_pos, 'is_hs300', [
        (d, s) in hs300_lookup for d, s in zip(df_factor2['TradingDate'], df_factor2['Symbol'])
    ])

# 4. 验证标记结果
counts = df_factor2['is_hs300'].value_counts()
print(f"✅ 标签添加完成！")
print(f"📊 每日处于沪深300名单内的记录数: {counts.get(True, 0)}")
print(f"📊 作为'非成分股'保留的历史记录数: {counts.get(False, 0)}")

# 检查一下某一天的数据是否准确
sample_date = df_300['TradingDate'].iloc[0]
print(f"\n检查日期 {sample_date} 的标记情况：")
print(df_factor2[df_factor2['TradingDate'] == sample_date]['is_hs300'].value_counts())

✅ 标签添加完成！
📊 每日处于沪深300名单内的记录数: 1060330
📊 作为'非成分股'保留的历史记录数: 1274039

检查日期 2010-01-05 00:00:00 的标记情况：
Series([], Name: count, dtype: int64)


In [6]:
# 按日期分组，并统计 is_hs300 列中 True 的数量
daily_true_counts = df_factor2.groupby('TradingDate')['is_hs300'].sum()

# 查看结果
print("每日成分股数量统计：")
print(daily_true_counts)

# 如果想看异常日期（比如数量不等于 300 的日子）
abnormal_days = daily_true_counts[daily_true_counts != 300]
if not abnormal_days.empty:
    print("\n⚠️ 发现成分股数量不等于 300 的日期：")
    print(abnormal_days)
else:
    print("\n✅ 所有日期的成分股数量均为 300。")

每日成分股数量统计：
TradingDate
2010-02-03    239
2010-02-04    285
2010-02-05    289
2010-02-08    287
2010-02-09    289
             ... 
2024-12-24    299
2024-12-25    299
2024-12-26    299
2024-12-27    299
2024-12-30    299
Name: is_hs300, Length: 3620, dtype: int64

⚠️ 发现成分股数量不等于 300 的日期：
TradingDate
2010-02-03    239
2010-02-04    285
2010-02-05    289
2010-02-08    287
2010-02-09    289
             ... 
2024-12-24    299
2024-12-25    299
2024-12-26    299
2024-12-27    299
2024-12-30    299
Name: is_hs300, Length: 2687, dtype: int64


In [ ]:
df_factor2.to_parquet('final_feature_matrix_polished2.parquet', index=False)
df_factor2